In [3]:
# LEVEL 1 — JUST THE MODEL
import torch
import torch.nn as nn

class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_size)
    self.fc = nn.Linear(hidden_size, num_classes)

  def forward(self,x):
    embedded = self.embedding(x)
    output,hidden = self.rnn(embedded)
    last = hidden[-1]
    output = self.fc(last)
    return output
model = SentimentRNN(vocab_size=2500, embedding_dim=50, hidden_size=256, num_classes=2)
dummy_inp = torch.randint(0, 1500, (10, 6))
output = model(dummy_inp)

print(output.shape)

torch.Size([6, 2])


In [11]:
# LEVEL 2 — TEXT → TENSOR
texts = [
    "i love this movie",
    "this film was terrible",
    "absolutely amazing performance",
    "worst film i have seen",
]

vocab = {"<PAD>":0, "<UNK>":1}
for text in texts:
  for word in text.split():
    if word not in vocab:
      vocab[word] = len(vocab)
print(vocab)

def encode(text, vocab, max_len=10):
  words = text.split()
  indices = [vocab.get(w,1) for w in words]
  indices = indices + [0] * (max_len - len(indices))
  indices = indices[:max_len]

  return indices
encoded = encode("i love this movie", vocab, max_len=10)
print(encoded)

x = torch.tensor([encoded], dtype=torch.long)
output = model(x)
print(output.shape)

{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'this': 4, 'movie': 5, 'film': 6, 'was': 7, 'terrible': 8, 'absolutely': 9, 'amazing': 10, 'performance': 11, 'worst': 12, 'have': 13, 'seen': 14}
[2, 3, 4, 5, 0, 0, 0, 0, 0, 0]
torch.Size([10, 2])


In [15]:
# LEVEL 3 — TRAINING LOOP
import torch.optim as optim
import torch.nn as nn # Ensure nn is imported if not already in this cell


class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, num_classes)

  def forward(self,x):
    embedded = self.embedding(x)
    output,hidden = self.rnn(embedded)
    last = hidden[-1]
    output = self.fc(last)
    return output

def encode(text, vocab, max_len=10):
  words = text.split()
  indices = [vocab.get(w,1) for w in words]
  # Pad if shorter than max_len
  if len(indices) < max_len:
    indices = indices + [0] * (max_len - len(indices))
  # Truncate if longer than max_len
  indices = indices[:max_len]
  return indices

texts = [
    "i love this movie",
    "this film was terrible",
    "amazing performance loved it",
    "worst film i have seen",
]
labels = [1, 0, 1, 0]
X = torch.tensor([encode(t, vocab, max_len=10) for t in texts], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)


model = SentimentRNN(vocab_size=len(vocab), embedding_dim=50, hidden_size=64, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
  model.train()
  predictions = model(X)
  loss = criterion(predictions, y)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  correct = (predictions.argmax(dim=1) == y).sum().item()
  print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Acc: {correct}/{len(y)}")

Epoch 1 | Loss: 0.7013 | Acc: 2/4
Epoch 2 | Loss: 0.6993 | Acc: 2/4
Epoch 3 | Loss: 0.6976 | Acc: 2/4
Epoch 4 | Loss: 0.6961 | Acc: 2/4
Epoch 5 | Loss: 0.6949 | Acc: 2/4
Epoch 6 | Loss: 0.6939 | Acc: 2/4
Epoch 7 | Loss: 0.6932 | Acc: 2/4
Epoch 8 | Loss: 0.6927 | Acc: 2/4
Epoch 9 | Loss: 0.6924 | Acc: 2/4
Epoch 10 | Loss: 0.6922 | Acc: 3/4


In [16]:
# LEVEL 4 — FULL PIPELINE
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=20):
        self.labels = labels
        self.data = [
            torch.tensor(encode(t, vocab, max_len), dtype=torch.long)
            for t in texts
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], torch.tensor(self.labels[idx], dtype=torch.long)


def train(model, loader, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0

        for X_batch, y_batch in loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            correct += (predictions.argmax(1) == y_batch).sum().item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Acc: {correct}/{len(dataset)}")


# Assemble
texts  = ["i love this", "terrible film", "amazing movie", "hated every minute"]
labels = [1, 0, 1, 0]

dataset    = TextDataset(texts, labels, vocab, max_len=10)
loader     = DataLoader(dataset, batch_size=2, shuffle=True)
model      = SentimentRNN(len(vocab), embedding_dim=50, hidden_size=64, num_classes=2)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), lr=0.001)

train(model, loader, criterion, optimizer, epochs=5)

Epoch 1 | Loss: 0.6984 | Acc: 2/4
Epoch 2 | Loss: 0.6984 | Acc: 2/4
Epoch 3 | Loss: 0.6934 | Acc: 2/4
Epoch 4 | Loss: 0.7160 | Acc: 2/4
Epoch 5 | Loss: 0.7241 | Acc: 2/4
